In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Clone your GitHub repo
!git clone https://github.com/amilabelhacini07-a11y/weakly-supervised-cancer-detection.git
%cd weakly-supervised-cancer-detection

# Configure Git
!git config --global user.name "Amila Belhacini"
!git config --global user.email "amilabelhacini07@gmail.com"

In [ ]:
!pip install torch torchvision timm scikit-learn matplotlib seaborn numpy pandas -q

In [ ]:
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Subset
import numpy as np

# Define transforms
transform = transforms.Compose([
    transforms.Resize((96, 96)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Save to Drive
DATA_PATH = "/content/drive/MyDrive/camelyon_project/data/"

# Download first (no transform yet — just download the files)
print("Downloading dataset...")
train_dataset = torchvision.datasets.PCAM(
    root=DATA_PATH,
    split='train',
    download=True,
    transform=transform
)

# ⚡ Use only a subset to avoid RAM crash
# 10,000 samples is more than enough for our experiments
TRAIN_SIZE = 10000
TEST_SIZE  = 2000

indices_train = np.random.choice(len(train_dataset), TRAIN_SIZE, replace=False)
indices_test  = np.random.choice(len(train_dataset), TEST_SIZE, replace=False)

train_subset = Subset(train_dataset, indices_train)
test_subset  = Subset(train_dataset, indices_test)

print(f"Training samples: {len(train_subset)}")
print(f"Test samples:     {len(test_subset)}")
print("✅ Dataset ready!")

In [ ]:
# Install dependencies
!pip install datasets timm grad-cam scikit-learn matplotlib seaborn -q

import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"RAM available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print("✅ All dependencies installed!")

In [ ]:
from datasets import load_dataset
import torch
from torch.utils.data import Dataset
import torchvision.transforms as transforms
import numpy as np
from PIL import Image

print("Loading PatchCamelyon dataset from HuggingFace...")

# Load 10,000 train + 2,000 test samples
train_data = load_dataset("1aurent/PatchCamelyon", split="train[:10000]")
test_data  = load_dataset("1aurent/PatchCamelyon", split="test[:2000]")

print(f"✅ Training samples: {len(train_data)}")
print(f"✅ Test samples:     {len(test_data)}")
print(f"Features: {train_data.features}")

In [ ]:
import matplotlib.pyplot as plt
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader

# ── Custom Dataset wrapper ──────────────────────────────────
class PCamDataset(Dataset):
    def __init__(self, hf_dataset, transform=None):
        self.dataset   = hf_dataset
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item  = self.dataset[idx]
        image = item['image'].convert('RGB')
        label = int(item['label'])
        if self.transform:
            image = self.transform(image)
        return image, label

# ── Transforms ──────────────────────────────────────────────
transform = transforms.Compose([
    transforms.Resize((96, 96)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# ── Create datasets & loaders ───────────────────────────────
train_dataset = PCamDataset(train_data, transform=transform)
test_dataset  = PCamDataset(test_data,  transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32,
                          shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=32,
                          shuffle=False, num_workers=2)

print(f"✅ Train batches: {len(train_loader)}")
print(f"✅ Test batches:  {len(test_loader)}")

# ── Visualise samples ───────────────────────────────────────
# Use raw images (no normalisation) for display
raw_transform = transforms.Compose([
    transforms.Resize((96, 96)),
    transforms.ToTensor()
])
raw_dataset = PCamDataset(train_data, transform=raw_transform)

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle('PatchCamelyon — Sample Patches\nTop: No Cancer | Bottom: Cancer',
             fontsize=14, fontweight='bold')

# Find 5 samples of each class
no_cancer, cancer = [], []
for i in range(500):
    _, label = raw_dataset[i]
    if label == 0 and len(no_cancer) < 5:
        no_cancer.append(i)
    elif label == 1 and len(cancer) < 5:
        cancer.append(i)
    if len(no_cancer) == 5 and len(cancer) == 5:
        break

for col in range(5):
    # No cancer row
    img, _ = raw_dataset[no_cancer[col]]
    axes[0, col].imshow(img.permute(1, 2, 0).numpy())
    axes[0, col].axis('off')
    axes[0, col].set_title('No Cancer', color='green', fontsize=9)

    # Cancer row
    img, _ = raw_dataset[cancer[col]]
    axes[1, col].imshow(img.permute(1, 2, 0).numpy())
    axes[1, col].axis('off')
    axes[1, col].set_title('Cancer', color='red', fontsize=9)

plt.tight_layout()

# Save to Drive
import os
RESULTS_PATH = "/content/drive/MyDrive/camelyon_project/results/"
os.makedirs(RESULTS_PATH, exist_ok=True)
plt.savefig(RESULTS_PATH + 'sample_images.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved to Google Drive!")

# ── Class distribution ──────────────────────────────────────
labels = [int(train_data[i]['label']) for i in range(len(train_data))]
n_cancer    = sum(labels)
n_no_cancer = len(labels) - n_cancer

plt.figure(figsize=(6, 4))
plt.bar(['No Cancer', 'Cancer'], [n_no_cancer, n_cancer],
        color=['#2ecc71', '#e74c3c'], edgecolor='black', width=0.5)
plt.title('Class Distribution — Training Set (10,000 samples)',
          fontweight='bold')
plt.ylabel('Count')
for i, v in enumerate([n_no_cancer, n_cancer]):
    plt.text(i, v + 50, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_PATH + 'class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nClass balance:")
print(f"  No Cancer: {n_no_cancer} ({n_no_cancer/len(labels)*100:.1f}%)")
print(f"  Cancer:    {n_cancer} ({n_cancer/len(labels)*100:.1f}%)")

In [ ]:
# Authentication cell - token removed for security
# When running: replace YOUR_TOKEN with your GitHub token
TOKEN = "your_token_here"
REPO  = "amilabelhacini07-a11y/weakly-supervised-cancer-detection"


In [ ]:
import shutil
import os

# Create folders first
os.makedirs('/content/weakly-supervised-cancer-detection/results', exist_ok=True)
os.makedirs('/content/weakly-supervised-cancer-detection/notebooks', exist_ok=True)

# Now copy the images
shutil.copy(
    '/content/drive/MyDrive/camelyon_project/results/sample_images.png',
    '/content/weakly-supervised-cancer-detection/results/sample_images.png'
)
shutil.copy(
    '/content/drive/MyDrive/camelyon_project/results/class_distribution.png',
    '/content/weakly-supervised-cancer-detection/results/class_distribution.png'
)

print("✅ Files copied!")
print(os.listdir('/content/weakly-supervised-cancer-detection/results'))

In [ ]:
# Authentication cell - token removed for security
# When running: replace YOUR_TOKEN with your GitHub token
TOKEN = "your_token_here"
REPO  = "amilabelhacini07-a11y/weakly-supervised-cancer-detection"


In [ ]:
###ResNet50 training cell###

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import timm
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import numpy as np
import time

# ── Device ──────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")

# ── Model ────────────────────────────────────────────────────
model = timm.create_model('resnet50', pretrained=True, num_classes=2)
model = model.to(device)
print("✅ ResNet50 loaded with pretrained ImageNet weights")

# ── Loss & Optimizer ─────────────────────────────────────────
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

# ── Training function ────────────────────────────────────────
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct    += (outputs.argmax(1) == labels).sum().item()
        total      += labels.size(0)
    return total_loss / len(loader), correct / total

# ── Evaluation function ──────────────────────────────────────
def evaluate(model, loader):
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            outputs = torch.softmax(model(images), dim=1)
            all_probs.extend(outputs[:, 1].cpu().numpy())
            all_labels.extend(labels.numpy())
    auc      = roc_auc_score(all_labels, all_probs)
    preds    = (np.array(all_probs) > 0.5).astype(int)
    acc      = accuracy_score(all_labels, preds)
    cm       = confusion_matrix(all_labels, preds)
    tn, fp, fn, tp = cm.ravel()
    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)
    return auc, acc, sensitivity, specificity, all_probs, all_labels

# ── Training loop ────────────────────────────────────────────
EPOCHS = 5
train_losses, train_accs = [], []
val_aucs = []

print(f"\nTraining ResNet50 for {EPOCHS} epochs...")
print("-" * 55)

best_auc = 0
for epoch in range(EPOCHS):
    start = time.time()

    # Train
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)

    # Evaluate
    auc, acc, sens, spec, probs, labels_val = evaluate(model, test_loader)

    scheduler.step()
    elapsed = time.time() - start

    train_losses.append(train_loss)
    train_accs.append(train_acc)
    val_aucs.append(auc)

    if auc > best_auc:
        best_auc = auc
        # Save best model to Drive
        torch.save(model.state_dict(),
                   '/content/drive/MyDrive/camelyon_project/resnet50_best.pth')
        saved = "💾 saved"
    else:
        saved = ""

    print(f"Epoch {epoch+1}/{EPOCHS} | "
          f"Loss: {train_loss:.4f} | "
          f"Acc: {train_acc:.4f} | "
          f"AUC: {auc:.4f} | "
          f"Sens: {sens:.4f} | "
          f"Spec: {spec:.4f} | "
          f"{elapsed:.1f}s {saved}")

print("-" * 55)
print(f"\n🏆 Best AUC: {best_auc:.4f}")

# ── Plot training curves ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(range(1, EPOCHS+1), train_losses, 'b-o', linewidth=2)
axes[0].set_title('Training Loss', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(True)

axes[1].plot(range(1, EPOCHS+1), val_aucs, 'r-o', linewidth=2)
axes[1].set_title('Validation AUC', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('AUC')
axes[1].set_ylim([0.5, 1.0])
axes[1].grid(True)

plt.suptitle('ResNet50 Baseline — Training Curves', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/camelyon_project/results/resnet50_training_curves.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Training curves saved!")

In [ ]:
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt

# Generate ROC curve using best model
# Load best model first
model.load_state_dict(torch.load(
    '/content/drive/MyDrive/camelyon_project/resnet50_best.pth',
    map_location=device
))

# Get predictions
auc, acc, sens, spec, probs, labels_val = evaluate(model, test_loader)

# Plot ROC curve
fpr, tpr, thresholds = roc_curve(labels_val, probs)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='#e74c3c', linewidth=2,
         label=f'ResNet50 (AUC = {auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random classifier')
plt.fill_between(fpr, tpr, alpha=0.1, color='#e74c3c')
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
plt.ylabel('True Positive Rate (Sensitivity)', fontsize=12)
plt.title('ROC Curve — ResNet50 Baseline\nPatchCamelyon Cancer Detection',
          fontweight='bold', fontsize=13)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/camelyon_project/results/resnet50_roc.png',
            dpi=150, bbox_inches='tight')
plt.show()
print(f"✅ ROC curve saved!")
print(f"\nFinal Results — ResNet50:")
print(f"  AUC:         {auc:.4f}")
print(f"  Accuracy:    {acc:.4f}")
print(f"  Sensitivity: {sens:.4f}")
print(f"  Specificity: {spec:.4f}")

In [ ]:
# Authentication cell - token removed for security
# When running: replace YOUR_TOKEN with your GitHub token
TOKEN = "your_token_here"
REPO  = "amilabelhacini07-a11y/weakly-supervised-cancer-detection"


In [ ]:
### train ViT-B/16 ###

In [ ]:
import timm
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix
import numpy as np
import matplotlib.pyplot as plt
import time

# ── Load ViT-B/16 ────────────────────────────────────────────
model_vit = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=2)
model_vit = model_vit.to(device)
print("✅ ViT-B/16 loaded with pretrained weights")

# ── ViT needs 224x224 images — update transforms ─────────────
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

transform_vit = transforms.Compose([
    transforms.Resize((224, 224)),  # ViT requires 224x224
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

train_dataset_vit = PCamDataset(train_data, transform=transform_vit)
test_dataset_vit  = PCamDataset(test_data,  transform=transform_vit)

train_loader_vit = DataLoader(train_dataset_vit, batch_size=32,
                               shuffle=True,  num_workers=2)
test_loader_vit  = DataLoader(test_dataset_vit,  batch_size=32,
                               shuffle=False, num_workers=2)

# ── Loss & Optimizer ─────────────────────────────────────────
criterion_vit = nn.CrossEntropyLoss()
optimizer_vit = optim.Adam(model_vit.parameters(), lr=1e-4)
scheduler_vit = optim.lr_scheduler.StepLR(optimizer_vit, step_size=3, gamma=0.1)

# ── Training loop ────────────────────────────────────────────
EPOCHS = 5
train_losses_vit, val_aucs_vit = [], []

print(f"\nTraining ViT-B/16 for {EPOCHS} epochs...")
print("-" * 55)

best_auc_vit = 0
for epoch in range(EPOCHS):
    start = time.time()

    train_loss, train_acc = train_epoch(model_vit, train_loader_vit,
                                        optimizer_vit, criterion_vit)
    auc, acc, sens, spec, probs_vit, labels_vit = evaluate(model_vit, test_loader_vit)

    scheduler_vit.step()
    elapsed = time.time() - start

    train_losses_vit.append(train_loss)
    val_aucs_vit.append(auc)

    if auc > best_auc_vit:
        best_auc_vit = auc
        torch.save(model_vit.state_dict(),
                   '/content/drive/MyDrive/camelyon_project/vit_best.pth')
        saved = "💾 saved"
    else:
        saved = ""

    print(f"Epoch {epoch+1}/{EPOCHS} | "
          f"Loss: {train_loss:.4f} | "
          f"Acc: {train_acc:.4f} | "
          f"AUC: {auc:.4f} | "
          f"Sens: {sens:.4f} | "
          f"Spec: {spec:.4f} | "
          f"{elapsed:.1f}s {saved}")

print("-" * 55)
print(f"\n🏆 Best ViT AUC: {best_auc_vit:.4f}")

# ── Plot training curves ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(range(1, EPOCHS+1), train_losses_vit, 'b-o', linewidth=2)
axes[0].set_title('Training Loss', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(True)

axes[1].plot(range(1, EPOCHS+1), val_aucs_vit, 'r-o', linewidth=2)
axes[1].set_title('Validation AUC', fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('AUC')
axes[1].set_ylim([0.5, 1.0])
axes[1].grid(True)

plt.suptitle('ViT-B/16 — Training Curves', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/camelyon_project/results/vit_training_curves.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ ViT training curves saved!")

In [ ]:
from sklearn.metrics import roc_curve

# Load best ViT model
model_vit.load_state_dict(torch.load(
    '/content/drive/MyDrive/camelyon_project/vit_best.pth',
    map_location=device
))

# Get ViT predictions
auc_vit, acc_vit, sens_vit, spec_vit, probs_vit, labels_vit = evaluate(
    model_vit, test_loader_vit
)

# Load best ResNet50 predictions (already have them)
model_res = timm.create_model('resnet50', pretrained=False, num_classes=2)
model_res.load_state_dict(torch.load(
    '/content/drive/MyDrive/camelyon_project/resnet50_best.pth',
    map_location=device
))
model_res = model_res.to(device)
auc_res, acc_res, sens_res, spec_res, probs_res, labels_res = evaluate(
    model_res, test_loader
)

# ── Combined ROC curve ───────────────────────────────────────
fpr_res, tpr_res, _ = roc_curve(labels_res, probs_res)
fpr_vit, tpr_vit, _ = roc_curve(labels_vit, probs_vit)

plt.figure(figsize=(8, 7))
plt.plot(fpr_res, tpr_res, color='#3498db', linewidth=2,
         label=f'ResNet50  (AUC = {auc_res:.4f})')
plt.plot(fpr_vit, tpr_vit, color='#e74c3c', linewidth=2,
         label=f'ViT-B/16  (AUC = {auc_vit:.4f})')
plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random classifier')
plt.fill_between(fpr_res, tpr_res, alpha=0.05, color='#3498db')
plt.fill_between(fpr_vit, tpr_vit, alpha=0.05, color='#e74c3c')
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=12)
plt.ylabel('True Positive Rate (Sensitivity)', fontsize=12)
plt.title('ROC Curves — ResNet50 vs ViT-B/16\nPatchCamelyon Cancer Detection',
          fontweight='bold', fontsize=13)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/camelyon_project/results/comparison_roc.png',
            dpi=150, bbox_inches='tight')
plt.show()

# ── Bar chart comparison ─────────────────────────────────────
metrics     = ['AUC', 'Accuracy', 'Sensitivity', 'Specificity']
resnet_vals = [auc_res, acc_res, sens_res, spec_res]
vit_vals    = [auc_vit, acc_vit, sens_vit, spec_vit]

x    = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, resnet_vals, width, label='ResNet50',
               color='#3498db', edgecolor='black')
bars2 = ax.bar(x + width/2, vit_vals,    width, label='ViT-B/16',
               color='#e74c3c', edgecolor='black')

for bar in bars1 + bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom',
            fontsize=9, fontweight='bold')

ax.set_ylabel('Score', fontsize=12)
ax.set_title('ResNet50 vs ViT-B/16 — Performance Comparison\nPatchCamelyon Cancer Detection',
             fontweight='bold', fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=11)
ax.set_ylim([0.6, 1.05])
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/camelyon_project/results/comparison_bar.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✅ Comparison plots saved!")

In [ ]:
# Authentication cell - token removed for security
# When running: replace YOUR_TOKEN with your GitHub token
TOKEN = "your_token_here"
REPO  = "amilabelhacini07-a11y/weakly-supervised-cancer-detection"


In [ ]:
# Authentication cell - token removed for security
# When running: replace YOUR_TOKEN with your GitHub token
TOKEN = "your_token_here"
REPO  = "amilabelhacini07-a11y/weakly-supervised-cancer-detection"
